# 12a WILDTRACK MVDeTr Training
Train MVDeTr on WILDTRACK, export ground-plane detections, convert them into repo-native tracklets and trajectories, and run ground-plane evaluation.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

WORK = Path('/kaggle/working')
PROJECT = WORK / 'gp'
MVDETR = WORK / 'MVDeTr'
DATA_HOME = Path.home() / 'Data'
WT_HOME = DATA_HOME / 'Wildtrack'
EXPORT_ROOT = PROJECT / 'data' / 'outputs' / 'wildtrack_mvdetr'

REPO_URL = 'https://github.com/MRKDaGods/gp.git'
if not PROJECT.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)])
else:
    subprocess.check_call(['git', '-C', str(PROJECT), 'pull', '--ff-only'])

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy<2', 'scipy', 'scikit-learn', 'opencv-python-headless',
    'kornia', 'motmetrics', 'matplotlib', 'pillow', 'lxml'
])
print(f'Repo ready at {PROJECT}')

In [ ]:
import subprocess, sys
# Pin kornia to version compatible with MVDeTr API (warp_perspective at top level)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'kornia==0.6.12'], check=False)

import numpy as np
np.float = np.float64
np.int = np.int_
np.bool = np.bool_

MVDETR_URL = 'https://github.com/hou-yz/MVDeTr.git'
if not MVDETR.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', MVDETR_URL, str(MVDETR)])
else:
    subprocess.check_call(['git', '-C', str(MVDETR), 'pull', '--ff-only'])

os.chdir(MVDETR)
req = MVDETR / 'requirements.txt'
if req.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
ops_dir = MVDETR / 'multiview_detector' / 'models' / 'ops'
mask_script = ops_dir / 'mask.sh'
make_script = ops_dir / 'make.sh'
build_failed = False
try:
    if mask_script.exists():
        subprocess.check_call(['bash', str(mask_script)], cwd=ops_dir)
    elif make_script.exists():
        subprocess.check_call(['bash', str(make_script)], cwd=ops_dir)
    else:
        raise FileNotFoundError(f'No MVDeTr build script found under {ops_dir}')
except subprocess.CalledProcessError as exc:
    build_failed = True
    print(f'WARNING: MVDeTr CUDA extension build failed ({exc}). Falling back to pure PyTorch ops.')

if build_failed:
    func_file = ops_dir / 'functions' / 'ms_deform_attn_func.py'
    module_file = ops_dir / 'modules' / 'ms_deform_attn.py'

    func_text = func_file.read_text(encoding='utf-8')
    if 'import MultiScaleDeformableAttention as MSDA' in func_text and 'except ImportError' not in func_text:
        func_text = func_text.replace(
            'import MultiScaleDeformableAttention as MSDA\n',
            'try:\n    import MultiScaleDeformableAttention as MSDA\nexcept ImportError:\n    MSDA = None\n',
            1,
        )
        func_file.write_text(func_text, encoding='utf-8')
        print(f'Patched fallback import: {func_file}')

    module_text = module_file.read_text(encoding='utf-8')
    if 'ms_deform_attn_core_pytorch' not in module_text:
        module_text = module_text.replace(
            'from ..functions import MSDeformAttnFunction\n',
            'from ..functions.ms_deform_attn_func import MSDeformAttnFunction, MSDA, ms_deform_attn_core_pytorch\n',
            1,
        )
        module_text = module_text.replace(
            '        output = MSDeformAttnFunction.apply(value, input_spatial_shapes, input_level_start_index,\n'
            '                                            sampling_locations, attention_weights, self.im2col_step)\n',
            '        if MSDA is None:\n'
            '            output = ms_deform_attn_core_pytorch(value, input_spatial_shapes, sampling_locations, attention_weights)\n'
            '        else:\n'
            '            output = MSDeformAttnFunction.apply(value, input_spatial_shapes, input_level_start_index,\n'
            '                                                sampling_locations, attention_weights, self.im2col_step)\n',
            1,
        )
        module_file.write_text(module_text, encoding='utf-8')
        print(f'Patched grid_sample fallback: {module_file}')

# Fix numpy and kornia compatibility in MVDeTr code
import re

compat_replacements = {
    'np.float': 'np.float64',
    'np.int': 'np.int_',
    'np.bool': 'np.bool_',
    'np.complex': 'np.complex128',
    'np.object': 'np.object_',
    'np.str': 'np.str_',
    'np.unicode': 'np.str_',
    'kornia.warp_perspective': 'kornia.geometry.transform.warp_perspective',
    'kornia.warp_affine': 'kornia.geometry.transform.warp_affine',
    'kornia.get_perspective_transform': 'kornia.geometry.transform.get_perspective_transform',
    'kornia.get_rotation_matrix2d': 'kornia.geometry.transform.get_rotation_matrix2d',
    'kornia.rgb_to_grayscale': 'kornia.color.rgb_to_grayscale',
}

for py_file in MVDETR.rglob('*.py'):
    text = py_file.read_text(encoding='utf-8')
    modified = False

    for old, new in compat_replacements.items():
        pattern = re.escape(old) + r'(?!\w)'
        if re.search(pattern, text):
            text = re.sub(pattern, new, text)
            modified = True

    if modified:
        py_file.write_text(text, encoding='utf-8')
        print(f'  Patched compat: {py_file.name}')

print(f'MVDeTr ready at {MVDETR}')

In [ ]:
def _find_wildtrack_mount():
    candidates = [
        Path('/kaggle/input/large-scale-multicamera-detection-dataset'),
        Path('/kaggle/input/datasets/aryashah2k/large-scale-multicamera-detection-dataset'),
        Path('/kaggle/input/wildtrack'),
        Path('/kaggle/input/datasets/aryashah2k/wildtrack'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def _best_effort_official_download():
    page = WORK / 'wildtrack_official.html'
    subprocess.run(['wget', '-q', '-O', str(page), 'https://www.epfl.ch/labs/cvlab/data/data-wildtrack/'], check=False)
    if not page.exists():
        return None
    html = page.read_text(encoding='utf-8', errors='ignore')
    urls = []
    for match in re.findall(r"href=[\"\']([^\"\']+)", html):
        if any(match.endswith(ext) for ext in ('.zip', '.tar', '.tar.gz', '.tgz')):
            if match.startswith('http'):
                urls.append(match)
            else:
                urls.append('https://www.epfl.ch' + match if match.startswith('/') else match)
    download_dir = WORK / 'wildtrack_official_download'
    download_dir.mkdir(exist_ok=True)
    for url in urls:
        target = download_dir / Path(url).name
        result = subprocess.run(['wget', '-q', '-O', str(target), url], check=False)
        if result.returncode == 0 and target.exists() and target.stat().st_size > 0:
            return download_dir
    return None

WILDTRACK_INPUT = _find_wildtrack_mount()
if WILDTRACK_INPUT is None:
    downloaded = _best_effort_official_download()
    if downloaded is None:
        raise FileNotFoundError('No mounted WILDTRACK dataset found and official download fallback did not resolve a usable archive. Attach a WILDTRACK Kaggle dataset.')
    WILDTRACK_INPUT = downloaded

print(f'WILDTRACK input root: {WILDTRACK_INPUT}')

In [ ]:
def _find_named_dir(root: Path, name: str):
    for candidate in [root / name, *root.rglob(name)]:
        if candidate.exists():
            return candidate
    return None

img_root = _find_named_dir(WILDTRACK_INPUT, "Image_subsets")
ann_root = _find_named_dir(WILDTRACK_INPUT, "annotations_positions")
cal_root = _find_named_dir(WILDTRACK_INPUT, "calibrations")
if img_root is None or ann_root is None or cal_root is None:
    raise FileNotFoundError("WILDTRACK dataset is missing Image_subsets, annotations_positions, or calibrations")

WT_HOME.parent.mkdir(parents=True, exist_ok=True)
if WT_HOME.exists() or WT_HOME.is_symlink():
    if WT_HOME.is_symlink() or WT_HOME.is_file():
        WT_HOME.unlink()
    else:
        shutil.rmtree(WT_HOME)
WT_HOME.mkdir(parents=True, exist_ok=True)
for name, src in {"Image_subsets": img_root, "annotations_positions": ann_root, "calibrations": cal_root}.items():
    dst = WT_HOME / name
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    try:
        dst.symlink_to(src, target_is_directory=True)
    except OSError:
        shutil.copytree(src, dst)

local_wt = PROJECT / "data" / "raw" / "wildtrack"
local_wt.mkdir(parents=True, exist_ok=True)
for name, src in {"Image_subsets": img_root, "annotations_positions": ann_root, "calibrations": cal_root}.items():
    dst = local_wt / name
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    try:
        dst.symlink_to(src, target_is_directory=True)
    except OSError:
        shutil.copytree(src, dst)

manifest_dir = local_wt / "manifests"
gt_dir = manifest_dir / "ground_truth"
if gt_dir.exists():
    shutil.rmtree(gt_dir)
gt_dir.mkdir(parents=True, exist_ok=True)

json_files = sorted((local_wt / "annotations_positions").glob("*.json"))
print(f"Preparing WILDTRACK ground truth from {len(json_files)} annotation frames")

FRAME_W, FRAME_H = 1920, 1080
MIN_VISIBLE_RATIO = 0.25
cam_tracks = {}
clipped_count = 0
skipped_count = 0

for frame_id, json_path in enumerate(json_files, start=1):
    with open(json_path, "r", encoding="utf-8") as handle:
        annotations = json.load(handle)

    for entry in annotations:
        person_id = entry["personID"]
        for view in entry.get("views", []):
            view_num = view["viewNum"]
            xmin = view.get("xmin", -1)
            ymin = view.get("ymin", -1)
            xmax = view.get("xmax", -1)
            ymax = view.get("ymax", -1)

            if xmin < 0 or ymin < 0 or xmax <= xmin or ymax <= ymin:
                continue

            orig_area = (xmax - xmin) * (ymax - ymin)
            xmin_c = max(0, xmin)
            ymin_c = max(0, ymin)
            xmax_c = min(FRAME_W, xmax)
            ymax_c = min(FRAME_H, ymax)

            if xmax_c <= xmin_c or ymax_c <= ymin_c:
                skipped_count += 1
                continue

            clipped_area = (xmax_c - xmin_c) * (ymax_c - ymin_c)
            if clipped_area / orig_area < MIN_VISIBLE_RATIO:
                skipped_count += 1
                continue

            if (xmin_c, ymin_c, xmax_c, ymax_c) != (xmin, ymin, xmax, ymax):
                clipped_count += 1

            camera_id = f"C{view_num + 1}"
            cam_tracks.setdefault(camera_id, {}).setdefault(person_id, []).append(
                (frame_id, person_id, xmin_c, ymin_c, xmax_c - xmin_c, ymax_c - ymin_c, 1.0, 0)
            )

if clipped_count:
    print(f"  Clipped {clipped_count} bboxes to frame boundaries ({FRAME_W}x{FRAME_H})")
if skipped_count:
    print(f"  Skipped {skipped_count} bboxes (<{MIN_VISIBLE_RATIO:.0%} visible or off-screen)")

for camera_id, tracks in sorted(cam_tracks.items()):
    rows = []
    for person_id, detections in sorted(tracks.items()):
        rows.extend(detections)
    rows.sort(key=lambda row: (row[0], row[1]))

    gt_path = gt_dir / f"{camera_id}.txt"
    with open(gt_path, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(",".join(str(value) for value in row) + "\n")

    print(f"  {camera_id}: {len(tracks)} identities, {len(rows)} detections -> {gt_path.name}")

print(f"Prepared upstream root: {WT_HOME}")
print(f"Prepared repo root: {local_wt}")
print(f"Prepared MOT ground truth: {gt_dir}")

In [ ]:
os.chdir(MVDETR)
EPOCHS = 25
TRAIN_ARGS = [
    sys.executable, 'main.py',
    '-d', 'wildtrack',
    '--arch', 'resnet18',
    '--world_feat', 'deform_trans',
    '--use_mse', 'false',
    '--epochs', str(EPOCHS),
    '--batch_size', '1',
    '--num_workers', '2',
    '--lr', '7e-4',
    '--world_reduce', '4',
    '--world_kernel_size', '10',
    '--img_reduce', '12',
    '--img_kernel_size', '10',
    '--dropout', '0.0',
    '--dropcam', '0.0',
]
print('Running:', ' '.join(TRAIN_ARGS))
subprocess.check_call(TRAIN_ARGS, cwd=MVDETR)

In [ ]:
log_root = MVDETR / 'logs' / 'wildtrack'
run_dirs = sorted([p for p in log_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
if not run_dirs:
    raise FileNotFoundError(f'No MVDeTr run directories found under {log_root}')
LATEST_RUN = run_dirs[-1]
TEST_TXT = LATEST_RUN / 'test.txt'
CKPT = LATEST_RUN / 'MultiviewDetector.pth'
if not TEST_TXT.exists():
    raise FileNotFoundError(f'Missing MVDeTr detections: {TEST_TXT}')
if not CKPT.exists():
    raise FileNotFoundError(f'Missing MVDeTr checkpoint: {CKPT}')
print(f'Latest run: {LATEST_RUN.name}')
print(f'Checkpoint: {CKPT}')
print(f'Detections: {TEST_TXT}')

In [ ]:
import csv

os.chdir(PROJECT)

raw_detections_export = WORK / "test.txt"
raw_checkpoint_export = WORK / "MultiviewDetector.pth"
converted_dir = EXPORT_ROOT / LATEST_RUN.name
converted_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(TEST_TXT, raw_detections_export)
shutil.copy2(CKPT, raw_checkpoint_export)
shutil.copy2(TEST_TXT, converted_dir / "test.txt")
shutil.copy2(CKPT, converted_dir / "MultiviewDetector.pth")

WILDTRACK_CELL_SIZE_CM = 2.5
WILDTRACK_X_MIN_CM = -300.0
WILDTRACK_Y_MIN_CM = -900.0

def normalize_wildtrack_frame(frame_id):
    return frame_id // 5 if frame_id >= 5 else frame_id

def grid_to_world_cm(grid_x, grid_y):
    return (
        WILDTRACK_X_MIN_CM + grid_x * WILDTRACK_CELL_SIZE_CM,
        WILDTRACK_Y_MIN_CM + grid_y * WILDTRACK_CELL_SIZE_CM,
    )

detections_payload = []
for line in TEST_TXT.read_text(encoding="utf-8").splitlines():
    stripped = line.strip()
    if not stripped:
        continue
    values = [float(value) for value in stripped.replace(",", " ").split()]
    if len(values) < 3:
        continue
    raw_frame_id = int(values[0])
    x_grid = float(values[1])
    y_grid = float(values[2])
    x_world_cm, y_world_cm = grid_to_world_cm(x_grid, y_grid)
    detections_payload.append({
        "frame_id": normalize_wildtrack_frame(raw_frame_id),
        "raw_frame_id": raw_frame_id,
        "x_grid": x_grid,
        "y_grid": y_grid,
        "x_world_cm": x_world_cm,
        "y_world_cm": y_world_cm,
        "score": 1.0,
    })

csv_path = converted_dir / "ground_plane_detections.csv"
with csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["frame_id", "raw_frame_id", "x_grid", "y_grid", "x_world_cm", "y_world_cm", "score"],
    )
    writer.writeheader()
    writer.writerows(detections_payload)

json_path = converted_dir / "ground_plane_detections.json"
json_path.write_text(json.dumps(detections_payload, indent=2), encoding="utf-8")
shutil.copy2(csv_path, WORK / "ground_plane_detections.csv")
shutil.copy2(json_path, WORK / "ground_plane_detections.json")

tracklets_by_camera = {}
trajectories = []
conversion_mode = "raw_only"

try:
    from src.stage_wildtrack_mvdetr.pipeline import run_stage_wildtrack_mvdetr
except (ImportError, ModuleNotFoundError) as exc:
    print(f"Skipping repo conversion: {exc}")
else:
    tracklets_by_camera, trajectories = run_stage_wildtrack_mvdetr(
        detections_path=TEST_TXT,
        calibration_dir=PROJECT / "data" / "raw" / "wildtrack" / "calibrations",
        output_dir=converted_dir,
        fps=2.0,
        max_match_distance_cm=75.0,
        max_missed_frames=5,
        min_track_length=2,
    )
    conversion_mode = "repo_stage"
    for name in ["ground_plane_tracks.csv", "ground_plane_tracks.json", "global_trajectories.json"]:
        path = converted_dir / name
        if path.exists():
            shutil.copy2(path, WORK / name)

print(f"Copied checkpoint to: {raw_checkpoint_export}")
print(f"Copied detections to: {raw_detections_export}")
print(f"Saved fallback CSV to: {csv_path}")
print(f"Saved fallback JSON to: {json_path}")
print(f"Conversion mode: {conversion_mode}")
print(f"Converted output dir: {converted_dir}")
print(f"Fallback detections exported: {len(detections_payload)}")
print(f"Projected tracklets: {sum(len(v) for v in tracklets_by_camera.values())}")
print(f"Global trajectories: {len(trajectories)}")

In [ ]:
summary = None

try:
    from src.stage5_evaluation.ground_plane_eval import evaluate_wildtrack_ground_plane
except (ImportError, ModuleNotFoundError) as exc:
    print(f"Skipping evaluation (module not available): {exc}")
else:
    if not trajectories:
        print("Skipping evaluation (no converted trajectories available)")
    else:
        try:
            result = evaluate_wildtrack_ground_plane(
                trajectories=trajectories,
                annotations_dir=PROJECT / "data" / "raw" / "wildtrack" / "annotations_positions",
                calibrations_dir=PROJECT / "data" / "raw" / "wildtrack" / "calibrations",
                conf_threshold=0.25,
                match_threshold_cm=50.0,
                nms_radius_cm=50.0,
            )
        except Exception as exc:
            print(f"Skipping evaluation (evaluation failed): {exc}")
        else:
            summary = {
                "run_name": LATEST_RUN.name,
                "moda": result.mota,
                "idf1": result.idf1,
                "id_switches": result.id_switches,
                "details": result.details,
            }
            summary_path = converted_dir / "ground_plane_eval_summary.json"
            summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
            shutil.copy2(summary_path, WORK / "ground_plane_eval_summary.json")
            print(json.dumps(summary, indent=2))
            print(f"Saved summary to {summary_path}")